<a href="https://colab.research.google.com/github/smagadi/AIML/blob/master/GenAI/chatGPTPowerdTechanalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [126]:
!pip install phidata openai yfinance googlesearch-python
!pip install pycountry

import yfinance as yf
import pandas as pd
import plotly.graph_objects as go

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime
from google.colab import userdata
import openai
import os

In [127]:
key =userdata.get('OPENAI')

# Retrieve the API key from Colab Secrets
# Try setting the API key as an environment variable:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI')


In [128]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime

def plot_candlestick_with_indicators(ticker, start_date='2020-01-01', end_date=datetime.today().strftime('%Y-%m-%d')):
    """
    Creates a candlestick chart with SMA, EMA, Bollinger Bands, and VWAP indicators.

    Parameters:
    ticker (str): Stock symbol (e.g., 'AAPL')
    start_date (str): Start date in 'YYYY-MM-DD' format
    end_date (str): End date in 'YYYY-MM-DD' format
    """

    # Fetch stock data
    stock_data = yf.Ticker(ticker).history(start=start_date, end=end_date)

    # Check if data exists
    if stock_data.empty:
        raise ValueError(f"No data found for {ticker} between {start_date} and {end_date}")

    # Calculate 20-day SMA
    stock_data['SMA_20'] = stock_data['Close'].rolling(window=20).mean()

    # Calculate 20-day EMA
    stock_data['EMA_20'] = stock_data['Close'].ewm(span=20, adjust=False).mean()

    # Calculate Bollinger Bands
    stock_data['BB_Mid'] = stock_data['Close'].rolling(window=20).mean()
    stock_data['BB_Upper'] = stock_data['BB_Mid'] + (2 * stock_data['Close'].rolling(window=20).std())
    stock_data['BB_Lower'] = stock_data['BB_Mid'] - (2 * stock_data['Close'].rolling(window=20).std())

    # Calculate VWAP
    stock_data['VWAP'] = (stock_data['Volume'] * (stock_data['High'] + stock_data['Low'] + stock_data['Close']) / 3).cumsum() / stock_data['Volume'].cumsum()

    # Interpolate NaN values
    stock_data['SMA_20'] = stock_data['SMA_20'].interpolate(method='linear', limit_direction='both')
    stock_data['BB_Mid'] = stock_data['BB_Mid'].interpolate(method='linear', limit_direction='both')
    stock_data['BB_Upper'] = stock_data['BB_Upper'].interpolate(method='linear', limit_direction='both')
    stock_data['BB_Lower'] = stock_data['BB_Lower'].interpolate(method='linear', limit_direction='both')

    # Create candlestick chart
    fig = go.Figure(data=[go.Candlestick(
        x=stock_data.index,
        open=stock_data['Open'],
        high=stock_data['High'],
        low=stock_data['Low'],
        close=stock_data['Close'],
        name='Candlestick'
    )])

    # Add SMA line
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['SMA_20'], mode='lines', name='SMA 20', line=dict(color='blue', width=2)))

    # Add EMA line
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['EMA_20'], mode='lines', name='EMA 20', line=dict(color='orange', width=2)))

    # Add Bollinger Bands
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['BB_Upper'], mode='lines', name='Bollinger Upper', line=dict(color='green', dash='dash', width=1)))
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['BB_Lower'], mode='lines', name='Bollinger Lower', line=dict(color='red', dash='dash', width=1)))
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['BB_Mid'], mode='lines', name='Bollinger Middle', line=dict(color='black', width=1)))

    # Add VWAP line
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['VWAP'], mode='lines', name='VWAP', line=dict(color='purple', width=2)))

    # Customize layout
    fig.update_layout(
        title=f'{ticker} Candlestick Chart with Indicators ({start_date} to {end_date})',
        xaxis_title='Date',
        yaxis_title='Price (USD)',
        xaxis_rangeslider_visible=False,
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    # Show plot
    fig.show()
    # Return stock data
    return stock_data




In [129]:


def plot_candlestick_with_indicatorsPlus(ticker, start_date='2020-01-01', end_date=datetime.today().strftime('%Y-%m-%d')):
    """
    Creates a candlestick chart with SMA, EMA, Bollinger Bands, VWAP, RSI, and MACD indicators.
    """
    # Fetch stock data
    stock_data = yf.Ticker(ticker).history(start=start_date, end=end_date)

    if stock_data.empty:
        raise ValueError(f"No data found for {ticker} between {start_date} and {end_date}")

    # Calculate SMA, EMA, Bollinger Bands
    stock_data['SMA_20'] = stock_data['Close'].rolling(window=20).mean()
    stock_data['EMA_20'] = stock_data['Close'].ewm(span=20, adjust=False).mean()
    stock_data['BB_Mid'] = stock_data['Close'].rolling(window=20).mean()
    stock_data['BB_Upper'] = stock_data['BB_Mid'] + (2 * stock_data['Close'].rolling(window=20).std())
    stock_data['BB_Lower'] = stock_data['BB_Mid'] - (2 * stock_data['Close'].rolling(window=20).std())
    stock_data['VWAP'] = (stock_data['Volume'] * (stock_data['High'] + stock_data['Low'] + stock_data['Close']) / 3).cumsum() / stock_data['Volume'].cumsum()

    # Calculate RSI
    delta = stock_data['Close'].diff()
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, abs(delta), 0)
    avg_gain = pd.Series(gain).rolling(window=14).mean()
    avg_loss = pd.Series(loss).rolling(window=14).mean()
    rs = avg_gain / avg_loss
    stock_data['RSI'] = 100 - (100 / (1 + rs))

    # Calculate MACD
    ema_12 = stock_data['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = stock_data['Close'].ewm(span=26, adjust=False).mean()
    stock_data['MACD'] = ema_12 - ema_26
    stock_data['Signal_Line'] = stock_data['MACD'].ewm(span=9, adjust=False).mean()

    # Interpolate NaN values
    for col in ['SMA_20', 'BB_Mid', 'BB_Upper', 'BB_Lower', 'RSI', 'MACD', 'Signal_Line']:
        stock_data[col] = stock_data[col].interpolate(method='linear', limit_direction='both')

    # Create candlestick chart
    fig = go.Figure(data=[go.Candlestick(
        x=stock_data.index,
        open=stock_data['Open'],
        high=stock_data['High'],
        low=stock_data['Low'],
        close=stock_data['Close'],
        name='Candlestick'
    )])

    # Add SMA, EMA, Bollinger Bands
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['SMA_20'], mode='lines', name='SMA 20', line=dict(color='blue', width=2)))
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['EMA_20'], mode='lines', name='EMA 20', line=dict(color='orange', width=2)))
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['BB_Upper'], mode='lines', name='Bollinger Upper', line=dict(color='green', dash='dash')))
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['BB_Lower'], mode='lines', name='Bollinger Lower', line=dict(color='red', dash='dash')))

    # Add RSI as a subplot
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['RSI'], mode='lines', name='RSI', line=dict(color='purple')))

    # Add MACD and Signal Line as subplots
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['MACD'], mode='lines', name='MACD Line', line=dict(color='black')))
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['Signal_Line'], mode='lines', name='Signal Line', line=dict(color='red')))

    # Customize layout
    fig.update_layout(
        title=f'{ticker} Candlestick Chart with Indicators ({start_date} to {end_date})',
        xaxis_title='Date',
        yaxis_title='Price (USD)',
        template='plotly_white',
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    fig.show()
    return stock_data



In [130]:
def ask_gpt_for_analysis_gpt4o_mini(stock_ticker, recent_stock_data):
    """
    Asks GPT-4o Mini for a financial analysis of the given stock data.

    Parameters:
        - stock_ticker: The ticker symbol of the stock.
        - recent_stock_data: A DataFrame containing recent financial data.

    Returns:
        - GPT-4o Mini's response as a string.
    """

    # Prepare the prompt for GPT-4o Mini
    prompt = f"""
You are a financial analyst. Based on the following recent data for {stock_ticker}, provide:
1. A recommendation (Buy/Sell/Hold) with reasons.
2. Explain current market conditions with respect to the stock
3. A recommendation with target price with clear reasdons for short term(next 5 days ),medium term(1 month),lomg term(1 month above)
4. Print the current price also.

Here is the data:

{recent_stock_data.tail(20).to_string(index=False)}

Please analyze this data and provide your insights.
This data contains the following indicators along with OHLC data:
- SMA_20
- EMA_20
- BB_Mid
- BB_Upper
- BB_Lower
- VWAP
- RSI
- MACD
- Signal_Line
"""

    # Call the OpenAI API using GPT-4o Mini
    # Update: Use client.chat.completions.create instead of openai.ChatCompletion.create
    # and import OpenAI
    from openai import OpenAI
    client = OpenAI()

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",  # Specify GPT-4o Mini as the model #gpt-4o-mini
        messages=[
            {"role": "system", "content": "You are a financial analyst providing actionable insights."},
            {"role": "user", "content": prompt}
        ]
    )

    # Extract and return GPT's response
    # Update: Access the content from the new response structure
    return response.choices[0].message.content

In [135]:
# Example usage
ticker="INFY.NS"
sd= plot_candlestick_with_indicators(ticker, '2024-09-01')

In [132]:

sd= plot_candlestick_with_indicatorsPlus(ticker, '2024-09-01')

In [137]:
# Ask GPT-4o Mini for analysis based on recent data
gpt_response = ask_gpt_for_analysis_gpt4o_mini(ticker, sd)

print("\nGPT Mini Analysis:")
print(gpt_response)


GPT Mini Analysis:
Based on the provided data and indicators, here are the insights and recommendations:

1. Recommendation: Hold
   - Reasons: The stock price of INFY.NS has been fluctuating within a range over the past few days. The indicators do not show a clear trend in either direction. It is advisable to hold the stock for now and wait for more definitive signals.

2. Current Market Conditions:
   - The stock has been trading within a range, with no clear trend in the short term. The volatility seems to be relatively low, as seen from the Bollinger Bands. The stock is hovering around the VWAP, indicating a fair valuation at the moment.

3. Recommendations and Target Prices:
   - Short Term (Next 5 days): Hold
     - Target Price: Range-bound trading expected around the current VWAP level.
   - Medium Term (1 month): Hold
     - Target Price: Expect the stock to continue trading within the current range, with potential upside if a clear trend develops.
   - Long Term (1 month abo